假设偶极子的磁矩为 $\vec{m}=m\hat{k}$ ，两个偶极子分别位于 $\left(a, 0, 0\right)$ 和 $\left(-a, 0, 0\right)$ ，考虑到磁层顶距离地心约 $10R_E$ ， $a \approx 10R_E$ 。

单个磁偶极子在空间一点 $\vec r$ 处产生的磁场为

$$
\vec B(\vec r)=\frac{\mu_0}{4\pi} \left[\frac{3(\vec m\cdot \vec R)\vec R}{R^5}-\frac{\vec m}{R^3}\right]
$$

其中：

$$
\vec R=\vec r-\vec r_0, \qquad R=|\vec R|
$$

定义

$$
R_{+}^2=(x-a)^2+y^2+z^2,
$$

$$
R_{-}^2=(x+a)^2+y^2+z^2,
$$

则各方向的磁场分量为：

$$
B_x=\frac{\mu_0m}{4\pi}3z\left[\frac{x-a}{R_{+}^{5}} + \frac{x+a}{R_{-}^{5}}\right]
$$

$$
B_y=\frac{\mu_0m}{4\pi}3yz\left[\frac{1}{R_{+}^{5}} + \frac{1}{R_{-}^{5}}\right]
$$

$$
B_z=\frac{\mu_0m}{4\pi}\left[\frac{2z^2-(x-a)^2-y^2}{R_{+}^{5}} + \frac{2z^2-(x+a)^2-y^2}{R_{-}^{5}}\right]
$$

In [2]:
import numpy as np

In [ ]:
# 只需知道磁场分布，因此可以先将常数全部等于1，令Rs=1，a=10
# x范围-30~30，y范围-20~20，z范围-30~30

In [ ]:
x_range = np.arange(-30, 30, 0.5)
y_range = np.arange(-20, 20, 0.5)
z_range = np.arange(-30, 30, 0.5)

X, Y, Z = np.meshgrid(x_range, y_range, z_range, indexing='ij')
Bx = np.full_like(X, np.nan, dtype=float)
By = np.full_like(X, np.nan, dtype=float)
Bz = np.full_like(X, np.nan, dtype=float)

In [ ]:
# 两个偶极子的位置为 (+a, 0, 0) 和 (-a, 0, 0)
a = 10.0
exclusion_radius = 0.5  # 保留 R=0.5 的值，给半径 1 的种子提供插值缓冲层

# 距离平方；先筛选，再计算带有分母的磁场表达式
R_plus_sq = (X - a)**2 + Y**2 + Z**2
R_minus_sq = (X + a)**2 + Y**2 + Z**2
valid = (R_plus_sq >= exclusion_radius**2) & (R_minus_sq >= exclusion_radius**2)

# 只提取两个排除球之外的点，避免在偶极子中心除零
xv, yv, zv = X[valid], Y[valid], Z[valid]
R_plus_5 = R_plus_sq[valid]**2.5
R_minus_5 = R_minus_sq[valid]**2.5

# 公共系数 mu_0*m/(4*pi) 取 1；球内保留 NaN
Bx[valid] = 3*zv*((xv - a)/R_plus_5 + (xv + a)/R_minus_5)
By[valid] = 3*yv*zv*(1/R_plus_5 + 1/R_minus_5)
Bz[valid] = (
    (2*zv**2 - (xv - a)**2 - yv**2)/R_plus_5
    + (2*zv**2 - (xv + a)**2 - yv**2)/R_minus_5
)


## 三维磁力线：球面种子点与 PyVista 网格流线

在安装了 NumPy、PyVista 的 **py312** 内核中从上到下运行。SciPy 已安装，但这一版用 PyVista 自带的流线积分器，无须调用 SciPy。

下面 N 是**每个偶极子球面的种子数**，两个球共 2N 个点。Fibonacci 球面点近似按面积均匀分布；一般 N 下无法保证所有点之间的距离严格相等。

**两个半径的作用不同：** 网格计算保留 R≥0.5 的值，用于半径 1 附近的插值；种子仍严格位于 R=1。最终磁力线只显示球外部分，在第一次返回任一 R=1 球面时截断。0.5≤R<1 是插值缓冲区域。

In [ ]:
import pyvista as pv

# StructuredGrid 的点序要求第一维变化最快，因此采用 Fortran 展平顺序。
grid = pv.StructuredGrid(X, Y, Z)
grid.point_data['B'] = np.column_stack([
    Bx.ravel(order='F'), By.ravel(order='F'), Bz.ravel(order='F'),
])
grid.point_data['B_magnitude'] = np.linalg.norm(grid.point_data['B'], axis=1)

# 只保留八个顶点都有有效磁场的体单元，确保插值不接触 NaN。
cell_valid = np.ones(tuple(n - 1 for n in X.shape), dtype=bool)
for di in (0, 1):
    for dj in (0, 1):
        for dk in (0, 1):
            cell_valid &= valid[
                di:di + X.shape[0] - 1,
                dj:dj + X.shape[1] - 1,
                dk:dk + X.shape[2] - 1,
            ]
trace_grid = grid.extract_cells(np.flatnonzero(cell_valid.ravel(order='F')))
print(f'用于追踪的有效体单元：{trace_grid.n_cells:,}')

### 1. 在每个半径为 1 的球面上放置 N 个点

单位球面的面积元可以写为 dA=dφ dz，因此让局部 z 坐标等间隔分布，再让方位角按黄金角转动，可以避免经纬线交点在两极聚集。

局部坐标满足 x²+y²+z²=1，乘以半径并加上偶极子中心，得到实际坐标。

In [ ]:
N = 64  # 每个球面的种子数；先用较小的偶数观察
seed_radius = 1.0
centers = np.array([[a, 0.0, 0.0], [-a, 0.0, 0.0]])


def fibonacci_sphere(n, radius, center):
    if not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError('N 必须是正整数')
    if radius <= 0:
        raise ValueError('球半径必须大于 0')
    i = np.arange(n)
    local_z = 1.0 - 2.0*(i + 0.5)/n
    phi = i*np.pi*(3.0 - np.sqrt(5.0))
    rho = np.sqrt(1.0 - local_z**2)
    unit_points = np.column_stack([
        rho*np.cos(phi), rho*np.sin(phi), local_z,
    ])
    return np.asarray(center) + radius*unit_points


seed_groups = [fibonacci_sphere(N, seed_radius, c) for c in centers]
seed_points = np.vstack(seed_groups)
seeds = pv.PolyData(seed_points)
seeds.point_data['dipole_id'] = np.repeat(np.arange(2), N)
print(f'每个球面 {N} 个种子点，总共 {seeds.n_points} 个。')

### 2. 让 PyVista 从球面种子点追踪流线

先确认每个种子都在有效网格单元内。streamlines_from_source 在网格中插值磁场，并使用 Runge–Kutta 方法追踪；双向积分从每个种子分别沿磁场正向和反向出发。

这里积分步长的单位设置为实际长度，最大步长为 0.005 RE（保留赤道附近很短的球外分支），和网格间距 0.5 RE 是不同参数。减小积分步长并不能消除网格插值误差。

原始轨迹可能进入 R<1 的插值缓冲区域。下一单元将删除直接向球内走的一支，并把另一支截断在第一次返回任一 R=1 球面的位置。

In [ ]:
# 如果以后改变了网格间距、半径或中心位置，先检查所有种子是否仍可插值。
seed_cells = trace_grid.find_containing_cell(seed_points)
if np.any(seed_cells < 0):
    raise ValueError('有种子位于有效网格外；请细化网格或减小排除半径。')

raw_lines = trace_grid.streamlines_from_source(
    seeds,
    vectors='B',
    integration_direction='both',
    integrator_type=45,
    step_unit='l',
    initial_step_length=0.001,
    min_step_length=0.0001,
    max_step_length=0.005,
    max_length=200.0,
    max_steps=100000,
    terminal_speed=1e-10,
    max_error=1e-6,
    compute_vorticity=False,
)
print(f'双向积分得到 {raw_lines.n_lines} 条原始分支。')

In [ ]:
def trim_at_spheres(path):
    """保留球外分支，在线段首次进入任一球的位置截断。"""
    path = np.asarray(path, dtype=float).copy()
    if len(path) < 2:
        return None
    # VTK 输出坐标可能是单精度，将首点恢复为原来的精确球面种子。
    seed_id = np.argmin(np.sum((seed_points - path[0])**2, axis=1))
    path[0] = seed_points[seed_id]
    source_center = centers[seed_id // N]
    normal = path[0] - source_center
    if np.dot(path[1] - path[0], normal) <= 0:
        return None  # 这一支从起点直接进入球内

    for j in range(len(path) - 1):
        start, end = path[j], path[j + 1]
        delta = end - start
        aa = np.dot(delta, delta)
        if aa == 0:
            continue
        first_hit = None
        for center in centers:
            relative = start - center
            bb = 2*np.dot(relative, delta)
            cc = np.dot(relative, relative) - seed_radius**2
            discriminant = bb*bb - 4*aa*cc
            if discriminant < 0:
                continue
            # 较小的根对应从球外进入球内；忽略起点自身的 t=0 根。
            t = (-bb - np.sqrt(discriminant))/(2*aa)
            if 1e-8 < t <= 1.0:
                first_hit = t if first_hit is None else min(first_hit, t)
        if first_hit is not None:
            return np.vstack([path[:j + 1], start + first_hit*delta])
    return path  # 未碰到球面，可能因网格边界、最大长度或弱磁场而停止


paths = []
connectivity = raw_lines.lines
cursor = 0
while cursor < len(connectivity):
    count = int(connectivity[cursor])
    point_ids = connectivity[cursor + 1:cursor + 1 + count]
    path = trim_at_spheres(raw_lines.points[point_ids])
    if path is not None and len(path) >= 2:
        paths.append(path)
    cursor += count + 1

if not paths:
    raise RuntimeError('没有可显示的球外磁力线，请检查种子和网格。')

connections = []
offset = 0
for path in paths:
    connections.append(np.r_[len(path), np.arange(offset, offset + len(path))])
    offset += len(path)
field_lines = pv.PolyData(np.vstack(paths), lines=np.concatenate(connections))
print(f'保留 {field_lines.n_lines} 条球外曲线。')


### 3. 用 PyVista 显示球面、种子点和磁力线

左图放大观察 +a 处球面的种子点，右图显示两个偶极子的磁力线。两幅图都保持空间长度比例。蓝色曲线只表示轨迹，不用线条数量表示磁场强度。

show_plot 在桌面弹出可旋转的窗口，可拖动旋转、滚轮缩放。若要在 Notebook 中静态显示，可使用返回的 Plotter 并调用 show(jupyter_backend='static')，同时在创建时改为 notebook=True。

只改 N 时，从种子点单元开始重新运行即可；修改 a 或网格范围时，应从前面的网格磁场部分重新运行。

In [ ]:
def make_plotter(off_screen=False):
    plotter = pv.Plotter(
        shape=(1, 2), notebook=False, off_screen=off_screen,
        window_size=(1500, 750),
    )
    colors = ['#f59e0b', '#a78bfa']
    plotter.subplot(0, 0)
    plotter.set_background('#f8fafc')
    plotter.add_mesh(pv.Sphere(radius=seed_radius, center=centers[0],
                               theta_resolution=64, phi_resolution=64),
                     color='#cbd5e1', opacity=0.55)
    plotter.add_mesh(pv.PolyData(seed_groups[0]), color='#b45309',
                     point_size=9, render_points_as_spheres=True)
    plotter.add_text(f'Surface seeds: N = {N}, radius = 1', font_size=12, color='#0f172a')
    plotter.add_axes()
    plotter.view_isometric()
    plotter.reset_camera()

    plotter.subplot(0, 1)
    plotter.set_background('#f8fafc')
    plotter.add_mesh(field_lines, color='#2563eb', line_width=1.5)
    for j, center in enumerate(centers):
        plotter.add_mesh(pv.Sphere(radius=seed_radius, center=center,
                                   theta_resolution=64, phi_resolution=64),
                         color=colors[j], smooth_shading=True)
        plotter.add_mesh(pv.PolyData(seed_groups[j]), color='#0f172a',
                         point_size=4, render_points_as_spheres=True)
    plotter.add_text('Two dipoles: 3D magnetic field lines', font_size=12, color='#0f172a')
    plotter.show_grid(xtitle='x / RE', ytitle='y / RE', ztitle='z / RE',
                      color='#475569', font_size=10, n_xlabels=3, n_ylabels=3, n_zlabels=3)
    plotter.add_axes()
    plotter.view_isometric()
    plotter.camera_position = [(55, -85, 40), (0, 0, 0), (0, 0, 1)]
    plotter.reset_camera()
    return plotter

In [ ]:
# 在 py312 内核中运行本单元，弹出交互式三维窗口。
plotter = make_plotter()
plotter.show()

参考文档：[PyVista 网格流线](https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.streamlines_from_source)、[StructuredGrid](https://docs.pyvista.org/api/core/_autosummary/pyvista.StructuredGrid.html)。